# Module 08 — Observabilité (métriques + alerte 6 h)

Surveiller le worker ingest : Prometheus + état stale via Postgres.

In [1]:
from fastapi.testclient import TestClient

from presslake.api.app import create_app

client = TestClient(create_app())

/home/anthony-marais/Documents/data_project/.venv/lib/python3.14/site-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


## Étape 1 — Métriques Prometheus

In [2]:
r = client.get("/metrics")
print("status:", r.status_code)
lines = [ln for ln in r.text.splitlines() if ln.startswith("presslake_")]
print("\n".join(lines[:8]))

status: 200
presslake_poll_runs_total 0.0
presslake_poll_runs_created 1.788207750932391e+09
presslake_articles_ingested_total 0.0
presslake_articles_ingested_created 1.788207750932406e+09
presslake_parse_runs_total 0.0
presslake_parse_runs_created 1.7882077509324148e+09
presslake_articles_parsed_total 0.0
presslake_articles_parsed_created 1.7882077509324205e+09


## Étape 2 — Ops status (alerte 6 h)

In [3]:
r = client.get("/ops/status")
ops = r.json()
for k, v in ops.items():
    print(f"{k}: {v}")

last_write_at: 2026-08-31T19:09:49.130204Z
seconds_since_write: 4370
stale_threshold_seconds: 21600
stale: False
articles_total: 128
message: OK — ingest récent.


## Étape 3 — Métriques depuis Postgres (worker_runs)

Les jauges `presslake_worker_*` et `presslake_catalog_*` sont rafraîchies depuis Postgres à chaque GET `/metrics`. Un `poll` en CLI écrit dans `worker_runs` même si le serve tourne ailleurs.

In [4]:
from presslake.storage.postgres import get_connection
from presslake.observability.worker_runs import JOB_POLL, log_worker_run

with get_connection() as conn:
    log_worker_run(conn, JOB_POLL, new_items=1)
    conn.commit()

r = client.get("/metrics")
worker_lines = [ln for ln in r.text.splitlines() if "presslake_worker" in ln or "presslake_catalog" in ln]
print("\n".join(worker_lines[:10]))

# HELP presslake_catalog_articles_total Nombre total d'articles dans le catalogue Postgres.
# TYPE presslake_catalog_articles_total gauge
presslake_catalog_articles_total 128.0
# HELP presslake_catalog_articles Articles par statut pipeline (fetched, parsed, …).
# TYPE presslake_catalog_articles gauge
presslake_catalog_articles{status="parsed"} 128.0
# HELP presslake_catalog_stale 1 si aucune écriture catalogue depuis plus de PRESSLAKE_STALE_HOURS, sinon 0.
# TYPE presslake_catalog_stale gauge
presslake_catalog_stale 0.0
# HELP presslake_worker_runs_total Nombre total d'exécutions worker enregistrées (poll ou parse).


## Étape 4 — CLI

```bash
uv run presslake ops status   # exit 1 si stale
uv run presslake serve
curl http://127.0.0.1:8000/ops/status
```

Tuto : [`docs/modules/08-observabilite.md`](../docs/modules/08-observabilite.md)